In [1]:
import sys
sys.path.append("..")                                   # 상위 폴더의 config.py를 불러오기 위해 경로 추가
import pandas as pd
from google.cloud import bigquery
import pydata_google_auth
from config import PROJECT_ID, LOCATION, MAX_BYTES_BILLED

# BigQuery 연결 (처음 한 번 로그인하면 이후에는 저장된 인증 정보 재사용)
credentials = pydata_google_auth.get_user_credentials(["https://www.googleapis.com/auth/bigquery"])
client = bigquery.Client(project=PROJECT_ID, credentials=credentials, location=LOCATION,
                         default_query_job_config=bigquery.QueryJobConfig(maximum_bytes_billed=MAX_BYTES_BILLED))

def query(sql):
    """SQL 문자열을 BigQuery에서 실행하고 결과를 DataFrame으로 돌려줌"""
    return client.query(sql).to_dataframe(create_bqstorage_client=False)

pd.set_option("display.max_rows", 100)                  # 일별 결과(56행 이상)가 생략되지 않고 다 보이도록

In [2]:

def run_sql_file(path):
    with open(path, encoding = "utf-8") as f:
        sql = f.read()
    job = client.query(sql)
    job.result()
    print(f"완료: {path} (처리량 {job.total_bytes_processed / 1024**3:.2f}GB") 

run_sql_file("../sql/stg/stg_events.sql")


완료: ../sql/stg/stg_events.sql (처리량 1.12GB
